In [0]:
# Import PySpark functions for DataFrame operations and data type transformations
from pyspark.sql import functions as F
from pyspark.sql.types import StringType

In [0]:
# Load the products table from Unity Catalog and inspect its structure
# This table contains product metadata including category, sub-category, and identifiers

products_df = spark.table("agentic_catalog.agentic_schema.products")
print("Product table schema")
products_df.printSchema()
print("sample product data")
products_df.show(5)
print(f"\nTotal product:{products_df.count()}")
# products_df = products_df.withColumn("product_id", F.col("product_id").cast(StringType()))
# products_df.write.mode("overwrite"

Product table schema
root
 |-- product_id: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_sub_category: string (nullable = true)

sample product data
+--------------------+--------------------+-------------------+--------------------+
|          product_id|        product_name|   product_category|product_sub_category|
+--------------------+--------------------+-------------------+--------------------+
|9eee3547-9ca0-4fe...|BrownBox SwiftWat...|            Gadgets|          Smartwatch|
|bff42cfb-2730-490...|          SmartX Pro|Wearable Technology|          Smartwatch|
|3de8f0d9-7e14-467...|    StridePro Runner|     Men/Women/Kids|               Shoes|
|594ef1d3-3dfc-482...|Urban Explorer Ja...|     Men/Women/Kids|            Clothing|
|b739e42e-ba7d-40d...|Elegance Extendab...|          Furniture|        Dining Table|
+--------------------+--------------------+-------------------+--------------------+
on

In [0]:
# Load the product documentation table from Unity Catalog
# This table contains the extracted text content from PDF product manuals

product_docs = spark.table("agentic_catalog.agentic_schema.product_docs")
print("product_docs table schema")
product_docs.printSchema()
print("sample product data")
product_docs.show(5)
print(f"\nTotal product_docs:{product_docs.count()}")
# products_df = products_df.withColumn("product_id", F.col("product_id").cast(StringType()))
# products_df.write.mode("overwrite"

product_docs table schema
root
 |-- product_name: string (nullable = true)
 |-- product_doc: string (nullable = true)

sample product data
+--------------------+--------------------+
|        product_name|         product_doc|
+--------------------+--------------------+
|     AccountEase Pro|AccountEase Pro D...|
|       AccuBooks Pro|AccuBooks Pro: Co...|
|AcoustiWave AirBu...|AcoustiWave AirBu...|
| ActiveFit Elite Pro|ActiveFit Elite P...|
|Advanced Algebra_...|Product Overview\...|
+--------------------+--------------------+
only showing top 5 rows

Total product_docs:509


In [0]:
# Join products and product documentation tables on product_name
# This combines product metadata with their respective documentation content

joined_df = products_df.join(product_docs, on="product_name", how="inner")

print("joined table schema")
joined_df.printSchema()
print("sample joined data")
joined_df.show(5)
print(f"total number of records in joined table {joined_df.count()}") 


joined table schema
root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_sub_category: string (nullable = true)
 |-- product_doc: string (nullable = true)

sample joined data
+--------------------+--------------------+----------------+--------------------+--------------------+
|        product_name|          product_id|product_category|product_sub_category|         product_doc|
+--------------------+--------------------+----------------+--------------------+--------------------+
|     AccountEase Pro|498cd716-f5ad-4c5...|        Software|     Online Platform|AccountEase Pro D...|
|       AccuBooks Pro|54d5b355-5c1b-458...|        Software| Accounting Software|AccuBooks Pro: Co...|
|AcoustiWave AirBu...|8dfeb92d-fc46-4f0...|     Accessories|    Wireless Earbuds|AcoustiWave AirBu...|
| ActiveFit Elite Pro|1be746e9-859c-42a...|       Wearables|          Smartwatch|ActiveFit Elite P...|
|  Ai

In [0]:

# COMMAND ----------

# DBTITLE 1,Create indexed_doc column
# Create an indexed document column with XML-style tags for structured retrieval
# This format makes it easier for LLMs to parse and understand product information
indexed_df = joined_df.withColumn(
    "indexed_doc",
    F.concat(
        F.lit("<product_category>"),
        F.col("product_category"),
        F.lit("</product_category>\n"),
        F.lit("<product_sub_category>"),
        F.col("product_sub_category"),
        F.lit("</product_sub_category>\n"),
        F.lit("<product_name>"),
        F.col("product_name"),
        F.lit("</product_name>\n"),
        F.lit("<product_doc>\n"),
        F.col("product_doc"),
        F.lit("\n</product_doc>")
    )
)

print("✓ Indexed document column created")

# Display sample indexed document
print("\nSample indexed document (first 500 characters):")
sample_indexed = indexed_df.select("product_name", "indexed_doc").first()
print(f"\nProduct: {sample_indexed['product_name']}")
print(f"\nIndexed Doc:\n{sample_indexed['indexed_doc'][:500]}...")

✓ Indexed document column created

Sample indexed document (first 500 characters):

Product: AccountEase Pro

Indexed Doc:
<product_category>Software</product_category>
<product_sub_category>Online Platform</product_sub_category>
<product_name>AccountEase Pro</product_name>
<product_doc>
AccountEase Pro Documentation
Product Overview
AccountEase Pro is a robust online platform designed to simplify
personal account management for individuals and businesses alike.
With an intuitive interface, users can seamlessly manage their
accounts, reset passwords, and maintain high security standards.
Getting Started
Sign Up
Go t...


In [0]:
# Select final columns for the enriched product documentation table
# Keep only the essential fields needed for AI agent retrieval

final_df = indexed_df.select(
    "product_name",
    "product_id",
    "product_category",
    "product_sub_category",
    "indexed_doc"
)
    
final_df.printSchema()
print("sample final data")
final_df.show(5)
print(f"total number of records in final table {final_df.count()}"  )

root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_sub_category: string (nullable = true)
 |-- indexed_doc: string (nullable = true)

sample final data
+--------------------+--------------------+----------------+--------------------+--------------------+
|        product_name|          product_id|product_category|product_sub_category|         indexed_doc|
+--------------------+--------------------+----------------+--------------------+--------------------+
|     AccountEase Pro|498cd716-f5ad-4c5...|        Software|     Online Platform|<product_category...|
|       AccuBooks Pro|54d5b355-5c1b-458...|        Software| Accounting Software|<product_category...|
|AcoustiWave AirBu...|8dfeb92d-fc46-4f0...|     Accessories|    Wireless Earbuds|<product_category...|
| ActiveFit Elite Pro|1be746e9-859c-42a...|       Wearables|          Smartwatch|<product_category...|
|  AirPure Elite 7000|5023

In [0]:
# COMMAND ----------

# DBTITLE 1,Save to table
# Write the enriched product documentation to Unity Catalog as a Delta table
# This table will be used for vector embeddings and AI agent retrieval

# Define the target table name
target_table = "agentic_catalog.agentic_schema.product_docs_combined"

# Write the DataFrame to Unity Catalog as a managed Delta table
# Using 'overwrite' mode to replace if exists
final_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✓ Table saved successfully: {target_table}")
print(f"\nTotal records written: {spark.table(target_table).count()}")

✓ Table saved successfully: agentic_catalog.agentic_schema.product_docs_combined

Total records written: 550


In [0]:
# DBTITLE 1,Verify table
# Verify the saved table and enable Change Data Feed for tracking updates
# Change Data Feed allows downstream systems to capture incremental changes

# Read the table back and verify
verify_df = spark.table("agentic_catalog.agentic_schema.product_docs_combined")

print("Verification Results:")
print(f"Total records: {verify_df.count()}")
print("\nTable Schema:")
verify_df.printSchema()

# Display sample records
print("\nSample Records:")
display(verify_df.limit(5))

# COMMAND ----------

# DBTITLE 1,Summary
# MAGIC %md
# MAGIC ## Summary
# MAGIC
# MAGIC ✓ **Pipeline Complete!**
# MAGIC
# MAGIC We have successfully:
# MAGIC 1. Loaded products and product documentation from Unity Catalog
# MAGIC 2. Joined the tables on `product_name`
# MAGIC 3. Created indexed documents with XML-style formatting
# MAGIC 4. Saved the results to `agentic_catalog.agentic_schema.products_indexed`
# MAGIC
# MAGIC The indexed documents are now ready for:
# MAGIC - Vector embeddings generation
# MAGIC - LLM-based retrieval
# MAGIC - Semantic search applications
# MAGIC - AI agent workflows

# COMMAND ----------

spark.sql("""
ALTER TABLE agentic_catalog.agentic_schema.product_docs_combined
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

Verification Results:
Total records: 550

Table Schema:
root
 |-- product_name: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- product_sub_category: string (nullable = true)
 |-- indexed_doc: string (nullable = true)


Sample Records:


product_name,product_id,product_category,product_sub_category,indexed_doc
AccountEase Pro,498cd716-f5ad-4c55-94d1-38f78bcf16c2,Software,Online Platform,"Software Online Platform AccountEase Pro AccountEase Pro Documentation Product Overview AccountEase Pro is a robust online platform designed to simplify personal account management for individuals and businesses alike. With an intuitive interface, users can seamlessly manage their accounts, reset passwords, and maintain high security standards. Getting Started Sign Up Go to the AccountEase Pro website. Click on the 'Sign Up' button. Fill out the registration form with your name, email, and password. Confirm your registration via the verification email. Login Access the portal through the homepage. Enter your registered email and password. Click 'Login'. Navigation The dashboard features user-friendly tabs for quick access to your profile, settings, and support. Account Management Resetting Your Password Click on 'Forgot Password?' on the login page. Enter your email address and submit the form. Check your email for a reset link and follow the instructions. Ensure the link is not expired (valid for 24 hours). Updating Profile Information1. 2. 3. 4. 5. 6. 7. 8. 9. 10. 11. 1. 2. 3. 4. 5. 6. Navigate to 'My Profile'. Edit fields such as your name, email, and phone number. Save changes. Managing Security Settings Go to 'Security Settings' in your account menu. Enable two-factor authentication for enhanced security. Review login history and change your password regularly. Common Troubleshooting Problem: Cannot Reset Password Ensure the reset email hasn't gone to the spam folder. Verify the email you entered is correct and registered. If the link is not working, request a new reset link. Problem: Email Verification Issues Check for a verification email and follow the link inside. Resend the verification email if not received within a few minutes. Problem: Account Locked Accounts may lock after consecutive failed login attempts. Contact support to unlock your account. Advanced Features Custom Integrations Connect AccountEase Pro with third-party applications like Google Drive or Dropbox. Use API keys for seamless data flow. User Customization Personalize the dashboard theme and notification settings.7. 8. 9. 10. 11. 12. 13. 1. 2. 3. 4. 5. 6. 7. 8. 9. 10. 1. 2. 3. 4. 5. Support and Resources FAQs Visit our FAQ section for quick answers. Customer Support Reach out to support via live chat or email at support@accounteasepro.com. Useful Links Tutorial Videos: www.accounteasepro.com/tutorials User Forum: www.accounteasepro.com/forum For further assistance, refer to our detailed support guides and video tutorials available through the platform. Welcome to AccountEase Pro, where managing your account is made easy and secure.• • • • • • •"
AccuBooks Pro,54d5b355-5c1b-4584-b631-9994464117bd,Software,Accounting Software,"Software Accounting Software AccuBooks Pro AccuBooks Pro: Comprehensive User Documentation Table of Contents Introduction System Requirements Installation Guide Core Features Getting Started Troubleshooting Frequently Asked Questions T echnical Support Updates and Maintenance Compliance and Security 1. Introduction AccuBooks Pro is a cutting-edge accounting software designed for small to medium-sized businesses. With a user-friendly interface and robust functionality, it streamlines financial management tasks such as invoicing, ledger maintenance, tax compliance, payroll, and customized reporting. 2. System Requirements Operating System : Windows 10 or later, macOS 10.14 or later Processor : 1 GHz or faster Memory : 8 GB RAM Storage : 500 MB of available hard disk space Internet : Broadband connection for updates and cloud features 3. Installation Guide Step 1: Download the AccuBooks Pro installer from our official website.1. 2. 3. 4. 5. 6. 7. 8. 9. 10. • • • • • Step 2: Double-click the downloaded file to begin the installation. Step 3: Follow the on-screen instructions to com

DataFrame[]

In [0]:
# Enable Change Data Feed on the product_docs_combined table
# This allows tracking of all INSERT, UPDATE, and DELETE operations

spark.sql("""
ALTER TABLE agentic_catalog.agentic_schema.product_docs_combined
SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")